# Coleta de dados - SAEB 2023

Microdados do SAEB 2023, do INEP (dados abertos).

Tema: **segurança na escola**. Os alunos se sentem seguros na escola? O que está ligado a isso?

In [1]:
import os
import urllib.request
import zipfile
import pandas as pd

## Baixando os dados
Na primeira vez baixa o zip do INEP (uns 700 MB) e coloca as tabelas .csv na pasta `tabelas_csv`.

In [2]:
url = "https://download.inep.gov.br/microdados/microdados_saeb_2023.zip"

if not os.path.exists("tabelas_csv/TS_ESCOLA.csv"):
    if not os.path.exists("microdados_saeb_2023.zip"):
        print("baixando o zip do INEP... (demora um pouco)")
        # baixa com outro nome e so renomeia no final, assim um download pela metade nao conta
        urllib.request.urlretrieve(url, "baixando.zip")
        os.rename("baixando.zip", "microdados_saeb_2023.zip")

    # tira so os .csv de dentro do zip e joga na pasta tabelas_csv
    print("extraindo as tabelas .csv...")
    os.makedirs("tabelas_csv", exist_ok=True)
    z = zipfile.ZipFile("microdados_saeb_2023.zip")
    for nome in z.namelist():
        if nome.endswith(".csv"):
            with open("tabelas_csv/" + os.path.basename(nome), "wb") as f:
                f.write(z.read(nome))
    z.close()

print("tabelas .csv prontas!\n")

tabelas .csv prontas!



## Alunos do 9º ano
O arquivo é muito grande, então pegamos só as colunas que vamos usar (veja o `dicionario_de_dados.md`).

In [3]:
colunas = ["ID_ESCOLA", "ID_UF", "IN_PUBLICA", "ID_LOCALIZACAO", "IN_SITUACAO_CENSO", "PESO_ALUNO_MT",
           "TX_RESP_Q23d", "TX_RESP_Q22f", "TX_RESP_Q23c", "TX_RESP_Q23e", "TX_RESP_Q23h"]
alunos = pd.read_csv("tabelas_csv/TS_ALUNO_9EF.csv", sep=";", encoding="latin1", usecols=colunas)
print("tabela alunos 9 ano:", alunos.shape)

# so alunos declarados no Censo Escolar
alunos = alunos[alunos["IN_SITUACAO_CENSO"] == 1]
print("declarados no Censo:", alunos.shape)

tabela alunos 9 ano: (2502907, 11)
declarados no Censo: (2489289, 11)


## Valores vazios
Nas respostas do questionário, `.` é em branco e `*` é resposta nula. Contamos os dois junto com os vazios.

In [4]:
perguntas = [c for c in colunas if c.startswith("TX_")]
sem_resposta = alunos[perguntas].isna() | alunos[perguntas].isin([".", "*"])
print("porcentagem sem resposta:")
print((sem_resposta.mean() * 100).round(1))

porcentagem sem resposta:
TX_RESP_Q23d    19.5
TX_RESP_Q22f    19.6
TX_RESP_Q23c    19.7
TX_RESP_Q23e    19.7
TX_RESP_Q23h    19.2
dtype: float64


## Q23d: "Eu me sinto seguro(a) quando estou na escola"
A = concordo totalmente, B = concordo, C = discordo, D = discordo totalmente, . = branco, * = nulo

Aqui a porcentagem é sem o peso amostral. No `seguranca_escola.ipynb` usamos o peso (`PESO_ALUNO_MT`), por isso lá o número do Brasil fica um pouco menor (28,7%).

In [5]:
print("respostas da Q23d:")
print(alunos["TX_RESP_Q23d"].value_counts())

validas = alunos[alunos["TX_RESP_Q23d"].isin(["A", "B", "C", "D"])]
nao_seguro = validas["TX_RESP_Q23d"].isin(["C", "D"]).mean() * 100
print(f"\nrespostas validas: {len(validas):,}")
print(f"nao se sentem seguros (C ou D): {nao_seguro:.1f}%")

respostas da Q23d:
TX_RESP_Q23d
B    1004798
.     471598
C     450869
A     389321
D     159139
*      13564
Name: count, dtype: int64

respostas validas: 2,004,127
nao se sentem seguros (C ou D): 30.4%


## Diretores
Cada diretor responde sobre a violência na escola (roubo, tráfico, porte de arma, bullying etc.). A mesma escola aparece **uma vez para cada série avaliada**, então é preciso deixar uma linha por escola.

In [6]:
colunas_dir = ["ID_ESCOLA", "IN_PREENCHIMENTO_QUESTIONARIO"] + [f"TX_Q{i:03d}" for i in range(93, 103)] + ["TX_Q082", "TX_Q104", "TX_Q105", "TX_Q123", "TX_Q196"]
diretores = pd.read_csv("tabelas_csv/TS_DIRETOR.csv", sep=";", encoding="latin1", usecols=colunas_dir)
print("tabela diretores:", diretores.shape)

diretores = diretores[diretores["IN_PREENCHIMENTO_QUESTIONARIO"] == 1]
print("questionarios preenchidos:", len(diretores))
print("escolas diferentes:", diretores["ID_ESCOLA"].nunique())

diretores = diretores.drop_duplicates("ID_ESCOLA")
print("depois de deixar uma linha por escola:", diretores.shape)

print("\ntrafico de drogas na escola (A = nunca, B = poucas vezes, C = muitas vezes, D = sempre):")
print(diretores["TX_Q094"].value_counts())

tabela diretores: (107089, 17)
questionarios preenchidos: 89785
escolas diferentes: 61870
depois de deixar uma linha por escola: (61870, 17)

trafico de drogas na escola (A = nunca, B = poucas vezes, C = muitas vezes, D = sempre):
TX_Q094
A    57106
B     3872
C      383
D       74
Name: count, dtype: int64


## Professores
Cada professor também responde sobre episódios de violência, incluindo lesão corporal e tiroteio. Usamos os professores do 9º ano (`ID_SERIE = 9`).

In [7]:
colunas_prof = ["ID_ESCOLA", "ID_SERIE"] + [f"TX_Q{i}" for i in range(135, 148)]
professores = pd.read_csv("tabelas_csv/TS_PROFESSOR.csv", sep=";", encoding="latin1", usecols=colunas_prof)
print("tabela professores:", professores.shape)
print("\nprofessores por serie:")
print(professores["ID_SERIE"].value_counts())

professores = professores[(professores["ID_SERIE"] == 9) & professores["TX_Q137"].isin(["A", "B", "C", "D"])]
print("\nprofessores do 9o ano que responderam as perguntas de violencia:", len(professores))

print("\ntiroteio ou bala perdida na escola:")
print(professores["TX_Q147"].value_counts())

tabela professores: (411876, 15)

professores por serie:
ID_SERIE
9     176496
12    124858
5      94952
13     13958
2       1612
Name: count, dtype: int64

professores do 9o ano que responderam as perguntas de violencia: 128084

tiroteio ou bala perdida na escola:
TX_Q147
A    125110
B      2019
C       552
D       194
Name: count, dtype: int64


## Ligando as tabelas
O `ID_ESCOLA` é um código fictício (LGPD), mas é o mesmo nas tabelas de alunos, diretores e professores. Vemos quantas escolas dá para ligar, usando só escolas com pelo menos 10 alunos respondendo a Q23d.

In [8]:
respostas_por_escola = validas.groupby("ID_ESCOLA").size()
escolas = respostas_por_escola[respostas_por_escola >= 10].index
print("escolas com 10+ respostas da Q23d:", len(escolas))

com_diretor = escolas.isin(diretores["ID_ESCOLA"])
com_professor = escolas.isin(professores["ID_ESCOLA"])
print(f"com questionario do diretor: {com_diretor.sum()} ({com_diretor.mean() * 100:.0f}%)")
print(f"com professores do 9o ano: {com_professor.sum()} ({com_professor.mean() * 100:.0f}%)")
print(f"com os dois: {(com_diretor & com_professor).sum()}")

escolas com 10+ respostas da Q23d: 36388
com questionario do diretor: 30959 (85%)
com professores do 9o ano: 30180 (83%)
com os dois: 29017


A análise completa está no `seguranca_escola.ipynb`.